In [84]:
import pandas as pd
import json
import os
import re

# Get the current working directory
current_directory = os.getcwd()

# Set the new working directory
new_directory = '/Users/lzz/Documents/GitHub/IVF/data_cleaning/output/'
os.chdir(new_directory)

In [85]:
# Load data
patients = pd.read_csv("patients_clean.csv")
data = pd.read_csv("embryo_data_clean.csv")
ids = pd.read_csv("embryo_ids_clean.csv")
transfers = pd.read_csv("embryo_tranfers_clean.csv")
treatments = pd.read_csv("embryo_treatments_clean.csv")

In [86]:
# Ensure PatientIDx is of type string in all DataFrames
if 'PatientIDx' in data.columns:
    data['PatientIDx'] = data['PatientIDx'].astype(str)
if 'PatientIDx' in ids.columns:
    ids['PatientIDx'] = ids['PatientIDx'].astype(str)
if 'PatientIDx' in patients.columns:
    patients['PatientIDx'] = patients['PatientIDx'].astype(str)
if 'PatientIDx' in treatments.columns:
    treatments['PatientIDx'] = treatments['PatientIDx'].astype(str)
if 'PatientIDx' in transfers.columns:
    transfers['PatientIDx'] = transfers['PatientIDx'].astype(str)

# Merging data
merged_1 = pd.merge(data, ids, on='EmbryoID', how='left')
merged_2 = pd.merge(merged_1, patients, on='PatientIDx', how='left')
merged_3 = pd.merge(merged_2, treatments, on='PatientIDx', how='left')
merged_4 = pd.merge(merged_3, transfers, on='PatientIDx', how='left')

# Remove columns where all values are NaN
merged_4 = merged_4.dropna(axis=1, how='all')
# print(merged_4)

                            EmbryoID            EmbryoFate  InstrumentNumber  \
0       D2023.05.21_S00667_I4425_P-2              Transfer            4425.0   
1      D2023.07.15_S00775_I4425_P-11                 Avoid            4425.0   
2       D2022.07.30_S00136_I4425_P-8  FrozenEmbryoTransfer            4425.0   
3       D2024.03.06_S01306_I4425_P-1                Freeze            4425.0   
4       D2024.03.06_S01306_I4425_P-1                Freeze            4425.0   
...                              ...                   ...               ...   
19484   D2022.11.12_S00341_I4425_P-8                Freeze            4425.0   
19485   D2023.10.17_S01007_I4425_P-1                 Avoid            4425.0   
19486  D2024.02.02_S01270_I4425_P-14                 Avoid            4425.0   
19487  D2024.02.02_S01270_I4425_P-14                 Avoid            4425.0   
19488  D2023.07.29_S00809_I4425_P-10                Freeze            4425.0   

       Position  WellNumber Fertilizati

In [87]:
# Calculate the age and add a new column Age
merged_4['DateOfBirth'] = pd.to_datetime(merged_4['DateOfBirth'], format='%Y.%m.%d')
merged_4['FertilizationTime'] = pd.to_datetime(merged_4['FertilizationTime'], format='%Y.%m.%d')
merged_4['Age'] = (merged_4['FertilizationTime'] - merged_4['DateOfBirth']).dt.days // 365

# Filter out rows where Age is less than 18
merged_4 = merged_4[merged_4['Age'] >= 18]

In [88]:
# Save the DataFrame to a CSV file
merged_4.to_csv('../output/merging_data.csv', index=False, encoding='utf-8-sig')